# GIAI ĐOẠN 0 – TV1
## Tìm hiểu và đánh giá chất lượng dữ liệu
**Đề tài:** Phân tích giỏ hàng siêu thị và xây dựng gợi ý mua kèm  
**Dataset:** `Groceries_dataset.csv`



## 1. Upload và đọc dữ liệu

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()

# Lấy tên file vừa upload
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Đã đọc file:", file_name)
print("Kích thước dữ liệu:", df.shape)
display(df.head())


## 2. Tìm hiểu cấu trúc dữ liệu

Dataset dự kiến có 3 thuộc tính:
- `Member_number`: mã khách hàng/thành viên.
- `Date`: ngày mua hàng.
- `itemDescription`: tên mặt hàng.


In [ ]:
print("===== THÔNG TIN CƠ BẢN =====")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
print("Tên các cột:", df.columns.tolist())

print("\n===== KIỂU DỮ LIỆU =====")
print(df.dtypes)

print("\n===== SỐ GIÁ TRỊ KHÁC NHAU =====")
print(df.nunique())


In [ ]:
# Xem thông tin chi tiết về kiểu dữ liệu và số giá trị non-null
df.info()


## 3. Kiểm tra dữ liệu thiếu

In [ ]:
missing_table = pd.DataFrame({
    "So_luong_thieu": df.isnull().sum(),
    "Ty_le_thieu_%": (df.isnull().mean() * 100).round(2)
})

display(missing_table)

print("Tổng số giá trị thiếu:", int(df.isnull().sum().sum()))


## 4. Kiểm tra dữ liệu trùng

In [ ]:
duplicate_count = int(df.duplicated().sum())
duplicate_rate = duplicate_count / len(df) * 100

print("Số dòng trùng hoàn toàn:", duplicate_count)
print(f"Tỷ lệ dòng trùng: {duplicate_rate:.2f}%")

# Hiển thị một số trường hợp trùng
duplicates = df[df.duplicated(keep=False)].sort_values(
    by=["Member_number", "Date", "itemDescription"]
)

display(duplicates.head(20))


## 5. Kiểm tra kiểu dữ liệu và ngày không hợp lệ

In [ ]:
# Không thay đổi df gốc, chỉ tạo biến tạm để kiểm tra
date_check = pd.to_datetime(
    df["Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

invalid_date_count = int(date_check.isna().sum())

print("Kiểu dữ liệu hiện tại của Date:", df["Date"].dtype)
print("Số ngày không hợp lệ:", invalid_date_count)

if invalid_date_count == 0:
    print("Tất cả giá trị Date đều có thể chuyển sang datetime.")
    print("Ngày nhỏ nhất:", date_check.min().date())
    print("Ngày lớn nhất:", date_check.max().date())
else:
    print("Các dòng có ngày không hợp lệ:")
    display(df[date_check.isna()].head(20))


## 6. Kiểm tra mã khách hàng

In [ ]:
invalid_member_mask = (
    df["Member_number"].isna() |
    (df["Member_number"] <= 0)
)

invalid_member_count = int(invalid_member_mask.sum())

print("Số Member_number không hợp lệ:", invalid_member_count)
print("Số khách hàng khác nhau:", df["Member_number"].nunique())
print("Member_number nhỏ nhất:", df["Member_number"].min())
print("Member_number lớn nhất:", df["Member_number"].max())


## 7. Kiểm tra tên mặt hàng

In [ ]:
item = df["itemDescription"]

# Tên mặt hàng rỗng hoặc chỉ có khoảng trắng
empty_item_mask = item.isna() | (item.astype(str).str.strip() == "")
empty_item_count = int(empty_item_mask.sum())

# Khoảng trắng ở đầu/cuối
item_as_text = item.astype(str)
space_issue_mask = item_as_text != item_as_text.str.strip()
space_issue_count = int(space_issue_mask.sum())

print("Số tên mặt hàng rỗng:", empty_item_count)
print("Số dòng có khoảng trắng đầu/cuối:", space_issue_count)

if space_issue_count > 0:
    print("\nCác tên mặt hàng có khoảng trắng đầu/cuối:")
    display(
        df.loc[space_issue_mask, ["itemDescription"]]
          .drop_duplicates()
          .head(20)
    )


In [ ]:
# Kiểm tra số tên mặt hàng trước và sau chuẩn hóa cơ bản
original_unique = item.nunique()
normalized_unique = (
    item.astype(str)
        .str.strip()
        .str.lower()
        .nunique()
)

print("Số tên mặt hàng ban đầu:", original_unique)
print("Số tên sau strip + lowercase:", normalized_unique)

if original_unique != normalized_unique:
    print("Có khả năng tồn tại các cách viết khác nhau của cùng một mặt hàng.")
else:
    print("Không phát hiện giảm số lượng tên mặt hàng sau chuẩn hóa cơ bản.")


## 8. Kiểm tra phân phối lệch của mặt hàng

Phần này chỉ dùng để **đánh giá chất lượng/phân bố dữ liệu**, không phải EDA đầy đủ của TV3.


In [ ]:
item_frequency = df["itemDescription"].value_counts()

print("===== THỐNG KÊ TẦN SUẤT MẶT HÀNG =====")
display(item_frequency.describe().to_frame("Tan_suat"))

print("\n10 mặt hàng xuất hiện nhiều nhất:")
display(item_frequency.head(10).to_frame("So_lan_xuat_hien"))

print("\nĐộ lệch (skewness) của phân phối tần suất:", round(item_frequency.skew(), 3))

top1_share = item_frequency.iloc[0] / len(df) * 100
top10_share = item_frequency.head(10).sum() / len(df) * 100

print(f"Tỷ lệ của mặt hàng phổ biến nhất: {top1_share:.2f}%")
print(f"Tỷ lệ của 10 mặt hàng phổ biến nhất: {top10_share:.2f}%")


## 9. Tổng hợp kết quả chất lượng dữ liệu

In [ ]:
quality_summary = pd.DataFrame({
    "Noi_dung_kiem_tra": [
        "Giá trị thiếu",
        "Dòng trùng hoàn toàn",
        "Ngày không hợp lệ",
        "Member_number không hợp lệ",
        "Tên mặt hàng rỗng",
        "Tên mặt hàng có khoảng trắng đầu/cuối"
    ],
    "So_luong": [
        int(df.isnull().sum().sum()),
        duplicate_count,
        invalid_date_count,
        invalid_member_count,
        empty_item_count,
        space_issue_count
    ]
})

quality_summary["Ty_le_tren_so_dong_%"] = (
    quality_summary["So_luong"] / len(df) * 100
).round(2)

display(quality_summary)
